# **NovaPay – Transaction Data Cleaning Notebook**

This notebook loads, inspects, and fully cleans the nova_pay_transcations.csv dataset:
- Fixes inconsistent categories (channel, countries, KYC tier)
- Converts columns to proper data types
- Handles missing values intelligently
- Handles duplicates and obvious outliers
- Saves a final cleaned CSV for modeling

**The Cleaning**

The merged raw file had 11,400 rows. I removed 200 duplicate transaction IDs, dropped 60 transactions with missing timestamps, and a few rows where both amount fields were missing, leaving about 11,137 valid transactions.
I standardized all the categorical text fields (countries, channels, KYC tiers), fixed common typos, and kept missing values in sensitive fields like IP, KYC tier, and home country because in fraud problems the absence of information itself is a risk signal.
For the numeric fields, I converted amounts to proper numeric types, handled invalid values, and imputed missing values based on currency-specific exchange rates and median fee ratios. I also imputed device trust scores by KYC tier, and enforced clean data types for all boolean and integer fields.
The final dataset has no missing values in the key numeric features, controlled missingness in a few categorical indicators, and is ready for feature engineering and modeling.

In [22]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

In [23]:
# Load raw dataset
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/NOVA PAY/nova_pay_merged_raw.csv"
df = pd.read_csv(DATA_PATH)
print("Initial shape:", df.shape)
df.head()


Initial shape: (11400, 26)


,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,device_id,new_device,ip_address,ip_country,location_mismatch,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,1.351351,9f292dcc-3297-4947-a260-6a1ef69041ff,False,221.78.171.180,US,False,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,12.758621,3a95b9f5-309f-4684-a46d-e2ff2435bf78,True,120.12.20.29,CA,False,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,7.142857,a4737752-9aac-43ed-9d8b-2ccdffc24052,False,223.96.181.93,US,False,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,0.925926,6aeb85a3-5603-4221-896c-9e6882764f1a,False,186.228.15.74,US,False,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,83.333333,a5b9250e-dbe0-4c5f-a6e7-5492b7349402,False,11.82.47.62,US,False,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

In [25]:
# Missing values
df.isna().sum().sort_values(ascending=False)

,0
amount_usd,305
ip_address,305
ip_country,301
kyc_tier,300
device_trust_score,295
fee,295
timestamp,29
transaction_id,0
customer_id,0
home_country,0


**2. Remove Duplicates**

In [26]:
# transaction_id must be unique in financial systems.
# Duplicate transactions are invalid & harmful for modeling.
before = df.shape[0]
df = df.drop_duplicates(subset="transaction_id", keep="first")
after = df.shape[0]

print(f"Removed {before - after} duplicate transaction_id rows")

Removed 200 duplicate transaction_id rows


**3. Fix Timestamp Column**

In [27]:
# Timestamps are required for fraud signals such as:
# - transaction velocity
# - time-of-day risk
# - date-based patterns
#
# Rows without timestamps cannot be used reliably.

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

missing_ts = df["timestamp"].isna().sum()
print("Missing timestamps:", missing_ts)

# Drop timestamp-missing rows (usually very few)
df = df[df["timestamp"].notna()].reset_index(drop=True)


Missing timestamps: 60


**4. Normalize Text Columns**

In [28]:
# Machine learning models treat different spellings as different categories.
# Example: "Web", "WEb", "webb", "weeb" → 4 categories → bad.
#
# Normalizing ensures:
# - fewer categories
# - stronger model generalization

def normalize_text(s):
    if pd.isna(s):
        return "unknown"
    s = str(s).strip().lower()
    if s == "" or s == "nan":
        return "unknown"
    return s

text_cols = ["home_country", "dest_currency", "source_currency", "channel", "kyc_tier", "ip_country"]

for col in text_cols:
    df[col] = df[col].apply(normalize_text)


**5. Fix Channel Typos**

In [29]:
#  "mobile", "web", "atm", "unknown".
# We unify common typos for consistency.

channel_map = {
    "mobille": "mobile",
    "mobil": "mobile",
    "m0bile": "mobile",
    "weeb": "web",
    "wb": "web",
    "at m": "atm",
    "atmm": "atm",
}

df["channel"] = df["channel"].replace(channel_map)


**6. Fix KYC Tier**

In [30]:
# KYC tiers matter for fraud risk.
# Uses clean categories: "low", "standard", "enhanced", "unknown".

kyc_map = {
    "standrd": "standard",
    "standard": "standard",
    "enhanced": "enhanced",
    "low": "low",
}

df["kyc_tier"] = df["kyc_tier"].map(kyc_map).fillna("unknown")


**7. Clean IP Address**

In [31]:
# Missing IP is not an error — it is a fraud pattern.
# We keep "unknown" as a valid category.

df["ip_address"] = df["ip_address"].fillna("unknown").astype(str).str.lower()


In [32]:
# Keep empty strings as NaN, but DO NOT convert "unknown" to NaN
for col in ["home_country", "ip_address", "ip_country", "kyc_tier"]:
    df[col] = df[col].replace(["", " "], np.nan)


**8. Fix Amount Columns Using Banking Logic**


*   amount_src missing → use amount_usd + conversion rate


*   amount_src <= 0 → invalid → treat as missing

*   both missing → drop row

*   amount_usd missing → derive from amount_src

In [33]:
# Convert amount_src to numeric
df["amount_src"] = pd.to_numeric(df["amount_src"], errors="coerce")

# Negative/zero values are invalid
df.loc[df["amount_src"] <= 0, "amount_src"] = np.nan

# Compute median conversion per currency
valid_amt = df[df["amount_src"].notna() & df["amount_usd"].notna()].copy()
conversion_rate = valid_amt.groupby("source_currency").apply(
    lambda g: (g["amount_usd"] / g["amount_src"]).median()
)

# Function to impute amount_src
def impute_amount_src(row):
    if pd.isna(row["amount_src"]) and pd.notna(row["amount_usd"]):
        rate = conversion_rate.get(row["source_currency"], np.nan)
        if pd.notna(rate):
            return row["amount_usd"] / rate
    return row["amount_src"]

df["amount_src"] = df.apply(impute_amount_src, axis=1)

# Drop rows where both amounts missing
df = df[~(df["amount_src"].isna() & df["amount_usd"].isna())]


/tmp/ipython-input-1792208417.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  conversion_rate = valid_amt.groupby("source_currency").apply(


In [34]:
# FIX amount_usd for rows that have amount_src but no USD

# Step 1 — Recompute conversion rate (median per currency)
valid_rows_usd = df[(df["amount_src"].notna()) & (df["amount_usd"].notna())]

conversion_rate_usd = (
    valid_rows_usd.groupby("source_currency")
    .apply(lambda g: (g["amount_usd"] / g["amount_src"]).median())
)

print("Conversion rates:\n", conversion_rate_usd)

# Step 2 — Impute missing amount_usd using conversion
def fix_amount_usd(row):
    if pd.isna(row["amount_usd"]) and pd.notna(row["amount_src"]):
        curr = row["source_currency"]
        rate = conversion_rate_usd.get(curr, np.nan)
        if pd.notna(rate):
            return row["amount_src"] * rate
    return row["amount_usd"]

df["amount_usd"] = df.apply(fix_amount_usd, axis=1)

# Step 3 — Backup fill with global rate (if any remain)
global_rate = (valid_rows_usd["amount_usd"] / valid_rows_usd["amount_src"]).median()
df["amount_usd"] = df["amount_usd"].fillna(df["amount_src"] * global_rate)

# Step 4 — As a final safety measure, fill any extreme edge case with median USD
df["amount_usd"] = df["amount_usd"].fillna(df["amount_usd"].median())

print("Missing amount_usd after fix:", df["amount_usd"].isna().sum())


/tmp/ipython-input-3676233781.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g["amount_usd"] / g["amount_src"]).median())


Conversion rates:
 source_currency
cad    0.74
gbp    1.25
usd    1.00
dtype: float64
Missing amount_usd after fix: 0


In [35]:
df["exchange_rate_src_to_dest"] = df["amount_usd"] / df["amount_src"]


In [36]:
df[col] = df[col].apply(lambda x: x.lower().strip() if isinstance(x,str) else x)


**9. Fix Fee Using Ratio Logic**

In [37]:
# Fees are usually proportional to amount.
# Uses median fee ratio per currency.

df["fee"] = pd.to_numeric(df["fee"], errors="coerce")
df.loc[df["fee"] < 0, "fee"] = np.nan

fee_valid = df[df["fee"].notna() & df["amount_usd"].notna()]
fee_ratio = (fee_valid["fee"] / fee_valid["amount_usd"]).median()

df["fee"] = df["fee"].fillna(df["amount_usd"] * fee_ratio)
df["fee"] = df["fee"].fillna(df["fee"].median())


**10. Fix Device Trust Score**

In [38]:
# Fix invalid device trust scores
df["device_trust_score"] = pd.to_numeric(df["device_trust_score"], errors="coerce")

# Replace negative values with NaN
df.loc[df["device_trust_score"] < 0, "device_trust_score"] = np.nan

# Impute by KYC tier median
median_dts = df.groupby("kyc_tier")["device_trust_score"].transform("median")
df["device_trust_score"] = df["device_trust_score"].fillna(median_dts)

# Final safety imputation
df["device_trust_score"] = df["device_trust_score"].fillna(df["device_trust_score"].median())


**11.  Enforce Final Data Types**

In [39]:
# Models need correct types (numeric vs categories).

df["new_device"] = df["new_device"].astype(bool)
df["location_mismatch"] = df["location_mismatch"].astype(bool)
df["is_fraud"] = df["is_fraud"].astype(int)

df["account_age_days"] = df["account_age_days"].astype(int)
df["txn_velocity_1h"] = df["txn_velocity_1h"].astype(int)
df["txn_velocity_24h"] = df["txn_velocity_24h"].astype(int)


**12. Final Check**

In [40]:
print("FINAL SHAPE:", df.shape)
df.isna().sum().sort_values(ascending=False)


FINAL SHAPE: (11137, 26)


,0
transaction_id,0
customer_id,0
timestamp,0
home_country,0
source_currency,0
dest_currency,0
channel,0
amount_src,0
amount_usd,0
fee,0


**13. Save the Cleaned Dataset**

In [41]:
OUTPUT_PATH = "nova_pay_transcations_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned data to: {OUTPUT_PATH}")


Saved cleaned data to: nova_pay_transcations_cleaned.csv
